# 02f · AWE real dataset (Fig. 7, Fig. 8)
6 clients, 2-layer bi-LSTM, 100 training epochs (rounds) + 50 recovery epochs, local lr 5e-5 (Adam), batch 32, 2 local epochs.
Compared: Retrain, FedSGA, FuGuard at unlearning lr ∈ {0.1, 0.05, 0.01}.

Fixes vs the repo: config keys; the repo ran with **no seed** (`CONFIG.get("seed")` → None); FedSGA's SGA loop in `main_awe/FedSGA.py` iterated
`train_dataset` (the previously trained client's data) instead of the target client's `target_dataset`; FuGuard encoded only 100 samples but kept all labels.
FuGuard trains its LSTM-VAE for 1000 epochs on the proxy subset *inside every unlearning call* (as in the repo), which is the slow part.
**Untested on the real data.**

In [ ]:
import os, sys, pathlib, importlib.util, subprocess
_cwd = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents, pathlib.Path("/content/FuGuard_Reproduction")]
             if (p / "fuguard_repro").exists()), None)
if ROOT is None:
    raise RuntimeError("Project not found. On Colab: run COLAB_START_HERE.ipynb first (it unzips to /content/FuGuard_Reproduction).")
os.environ.setdefault("FUGUARD_ROOT", str(ROOT))
sys.path.insert(0, str(ROOT))
for _mod in ("geomloss", "diffusers"):          # a Colab runtime restart wipes pip installs -> re-install if missing
    if importlib.util.find_spec(_mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
        break
print("project root:", ROOT)

In [ ]:
from fuguard_repro.config import lr_variant
from fuguard_repro.runner import run_grid

LRS = (0.1, 0.05, 0.01)
jobs = [("retrain", "retrain", {})]
for lr in LRS:
    jobs += [(lr_variant("fedsga", lr), "fedsga", {"forgetting_lr": lr}),
             (lr_variant("fuguard", lr), "fuguard", {"forgetting_lr": lr})]
run_grid("awe", ["AWE"], jobs, seeds=(0,), recovery_rounds=50, kind="awe")

In [ ]:
from IPython.display import Image, display
from fuguard_repro import plotting
for p in (plotting.plot_awe_training(), plotting.plot_awe_recovery(LRS)):
    if p: display(Image(str(p)))